# 01 — Preprocesamiento de datos

Cargar el dataset de PaySim, crear las variables del anexo 1 y separar el periodo de entrenamiento (step <= 400) del de producción (step >= 401).

La data viene limpia, no hace falta una limpieza profunda.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

SEED = 42
STEP_CORTE = 400
N_SPLITS = 5
TARGET = "isFraud"

# busco la raíz del proyecto por si se abre el notebook desde otra carpeta
BASE = Path.cwd()
for candidato in [BASE, *BASE.parents]:
    if (candidato / "Datos" / "PS_20174392719_1491204439457_log.csv").exists():
        RUTA_BASE = candidato
        break
else:
    raise FileNotFoundError("No encontré el CSV en Datos/")

RUTA_CSV = RUTA_BASE / "Datos" / "PS_20174392719_1491204439457_log.csv"
RUTA_PROC = RUTA_BASE / "Datos" / "procesado"
RUTA_PROC.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
print("Raíz del proyecto:", RUTA_BASE)

Raíz del proyecto: D:\DSRP-ML3-Proyecto-Final


## 1. Carga del dataset

In [2]:
df = pd.read_csv(RUTA_CSV)
print("Shape:", df.shape)
print("\nColumnas:", list(df.columns))
df.head()

Shape: (6362620, 11)

Columnas: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,"9,839.6400",C1231006815,"170,136.0000","160,296.3600",M1979787155,0.0000,0.0000,0,0
1,1,PAYMENT,"1,864.2800",C1666544295,"21,249.0000","19,384.7200",M2044282225,0.0000,0.0000,0,0
2,1,TRANSFER,181.0000,C1305486145,181.0000,0.0000,C553264065,0.0000,0.0000,1,0
3,1,CASH_OUT,181.0000,C840083671,181.0000,0.0000,C38997010,"21,182.0000",0.0000,1,0
4,1,PAYMENT,"11,668.1400",C2048537720,"41,554.0000","29,885.8600",M1230701703,0.0000,0.0000,0,0


## 2. Entrenamiento vs producción

step <= 400 es entrenamiento, step >= 401 es producción.

In [3]:
resumen = {}
resumen["filas_totales"] = int(len(df))
resumen["pct_fraude_global"] = round(float(100 * df[TARGET].mean()), 4)

df["segmento"] = np.where(df["step"] <= STEP_CORTE, "entrenamiento", "produccion")

tabla_segmentos = (
    df.groupby("segmento")[TARGET]
    .agg(filas="size", fraudes="sum", pct_fraude=lambda s: 100 * s.mean())
    .round(4)
)
resumen["segmentos"] = {
    k: {c: float(v) for c, v in fila.items()} for k, fila in tabla_segmentos.to_dict("index").items()
}
print(f"% de fraude global: {resumen['pct_fraude_global']}%")
tabla_segmentos

% de fraude global: 0.1291%


,filas,fraudes,pct_fraude
segmento,,,
entrenamiento,5787030,4477,0.0774
produccion,575590,3736,0.6491


## 3. Algunas comprobaciones del dataset

In [4]:
# ¿el fraude pasa en todos los tipos de transacción?
fraude_por_tipo = (
    df.groupby("type")[TARGET]
    .agg(filas="size", fraudes="sum", pct_fraude=lambda s: 100 * s.mean())
    .sort_values("pct_fraude", ascending=False)
    .round(4)
)
resumen["fraude_por_tipo"] = {
    k: {c: float(v) for c, v in fila.items()} for k, fila in fraude_por_tipo.to_dict("index").items()
}
fraude_por_tipo

,filas,fraudes,pct_fraude
type,,,
TRANSFER,532909,4097,0.7688
CASH_OUT,2237500,4116,0.1840
CASH_IN,1399284,0,0.0000
DEBIT,41432,0,0.0000
PAYMENT,2151495,0,0.0000


In [5]:
# el saldo debería cuadrar siempre: oldbalance - amount = newbalance
# si no cuadra, queda una diferencia (errorBalanceOrig)
df["errorBalanceOrig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]

tabla_error = (
    df.groupby(TARGET)["errorBalanceOrig"]
    .agg(media="mean", mediana="median", pct_inconsistente=lambda s: 100 * (s.abs() > 1).mean())
    .round(2)
)
resumen["errorBalanceOrig"] = {
    str(k): {c: float(v) for c, v in fila.items()} for k, fila in tabla_error.to_dict("index").items()
}
tabla_error

,media,mediana,pct_inconsistente
isFraud,,,
0,"-201,338.5600","-69,049.3100",78.7000
1,"-10,692.3300",0.0000,0.5500


In [6]:
# isFlaggedFraud es una regla del negocio (>200000), a ver cuántos fraudes atrapa
fraudes = df[df[TARGET] == 1]
recall_flag = float(fraudes["isFlaggedFraud"].mean())
resumen["isFlaggedFraud"] = {
    "fraudes_totales": int(len(fraudes)),
    "fraudes_marcados": int(fraudes["isFlaggedFraud"].sum()),
    "recall": round(100 * recall_flag, 4),
}
print(f"Fraudes totales:               {len(fraudes):,}")
print(f"Marcados por isFlaggedFraud:   {int(fraudes['isFlaggedFraud'].sum()):,}")
print(f"Recall de la regla:            {100 * recall_flag:.4f}%")

Fraudes totales:               8,213
Marcados por isFlaggedFraud:   16
Recall de la regla:            0.1948%


In [7]:
# ¿los destinatarios merchant (nameDest con M) tienen saldo?
es_merchant = df["nameDest"].str.startswith("M")
resumen["merchants"] = {
    "cuentas_merchant": int(es_merchant.sum()),
    "pct_del_dataset": round(float(100 * es_merchant.mean()), 4),
    "merchants_con_saldo_origen": int((df.loc[es_merchant, "oldbalanceDest"] > 0).sum()),
}
print(f"Transacciones hacia merchant: {resumen['merchants']['cuentas_merchant']:,} "
      f"({resumen['merchants']['pct_del_dataset']}% del dataset)")
print(f"Merchants con oldbalanceDest > 0: {resumen['merchants']['merchants_con_saldo_origen']:,}")
df.loc[es_merchant, ["oldbalanceDest", "newbalanceDest"]].describe()

Transacciones hacia merchant: 2,151,495 (33.8146% del dataset)
Merchants con oldbalanceDest > 0: 0


,oldbalanceDest,newbalanceDest
count,"2,151,495.0000","2,151,495.0000"
mean,0.0000,0.0000
std,0.0000,0.0000
min,0.0000,0.0000
25%,0.0000,0.0000
50%,0.0000,0.0000
75%,0.0000,0.0000
max,0.0000,0.0000


## 4. Variables nuevas (anexo 1)

In [8]:
# errorBalanceOrig ya está de la sección anterior, completo el resto
df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

# qué fracción del saldo inicial se mueve en la transacción
with np.errstate(divide="ignore"):
    df["ratio_amount"] = (
        (df["amount"] / df["oldbalanceOrg"])
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

df["isMerchant"] = df["nameDest"].str.startswith("M").astype(int)
df["isTransfer"] = (df["type"] == "TRANSFER").astype(int)
df["isCashOut"] = (df["type"] == "CASH_OUT").astype(int)

FEATURES = [
    "step", "amount", "oldbalanceOrg", "newbalanceOrig",
    "oldbalanceDest", "newbalanceDest",
    "errorBalanceOrig", "errorBalanceDest", "ratio_amount",
    "isMerchant", "isTransfer", "isCashOut",
]

# isFlaggedFraud y los nombres de cuenta no entran al modelo
df[FEATURES] = df[FEATURES].astype(np.float32)
print(f"{len(FEATURES)} features listas")
df[FEATURES].describe().T

12 features listas


,count,mean,std,min,25%,50%,75%,max
step,"6,362,620.0000",243.3972,142.3320,1.0000,156.0000,239.0000,335.0000,743.0000
amount,"6,362,620.0000","179,861.8906","603,858.2500",0.0000,"13,389.5703","74,871.9375","208,721.4805","92,445,520.0000"
oldbalanceOrg,"6,362,620.0000","833,883.1250","2,888,242.7500",0.0000,0.0000,"14,208.0000","107,315.1758","59,585,040.0000"
newbalanceOrig,"6,362,620.0000","855,113.6250","2,924,048.5000",0.0000,0.0000,0.0000,"144,258.4062","49,585,040.0000"
oldbalanceDest,"6,362,620.0000","1,100,701.7500","3,399,180.2500",0.0000,0.0000,"132,705.6641","943,036.6875","356,015,904.0000"
newbalanceDest,"6,362,620.0000","1,224,996.3750","3,674,129.0000",0.0000,0.0000,"214,661.4453","1,111,909.2188","356,179,264.0000"
errorBalanceOrig,"6,362,620.0000","-201,092.4844","606,650.4375","-92,445,520.0000","-249,641.0898","-68,677.2539","-2,954.2300",0.0100
errorBalanceDest,"6,362,620.0000","55,567.1719","441,528.7812","-75,885,728.0000",0.0000,"3,500.4900","29,353.0444","13,191,234.0000"
ratio_amount,"6,362,620.0000",81.7135,"3,395.7053",0.0000,0.0000,0.0768,2.2194,"3,925,475.5000"
isMerchant,"6,362,620.0000",0.3381,0.4731,0.0000,0.0000,0.0000,1.0000,1.0000


## 5. Split y StratifiedKFold

Guardo train y prod en Datos/procesado/ con la asignación de folds para el notebook de entrenamiento.

In [9]:
train = df.loc[df["step"] <= STEP_CORTE, FEATURES + [TARGET]].copy()
prod = df.loc[df["step"] > STEP_CORTE, FEATURES + [TARGET]].copy()

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
train["fold"] = -1
for k, (_, idx_val) in enumerate(skf.split(train[FEATURES], train[TARGET])):
    train.loc[train.index[idx_val], "fold"] = k

resumen["shape_train"] = list(train.shape)
resumen["shape_prod"] = list(prod.shape)
resumen["pct_fraude_train"] = round(float(100 * train[TARGET].mean()), 4)
resumen["pct_fraude_prod"] = round(float(100 * prod[TARGET].mean()), 4)
resumen["features"] = FEATURES

print(f"Train: {train.shape[0]:,} filas | fraude {resumen['pct_fraude_train']}%")
print(f"Prod:  {prod.shape[0]:,} filas | fraude {resumen['pct_fraude_prod']}%")
print("\nFraude por fold:")
display(train.groupby("fold")[TARGET].agg(filas="size", pct_fraude="mean").round(4))

Train: 5,787,030 filas | fraude 0.0774%
Prod:  575,590 filas | fraude 0.6491%

Fraude por fold:


,filas,pct_fraude
fold,,
0,1157406,0.0008
1,1157406,0.0008
2,1157406,0.0008
3,1157406,0.0008
4,1157406,0.0008


In [10]:
train.to_parquet(RUTA_PROC / "train.parquet", index=False)
prod.to_parquet(RUTA_PROC / "prod.parquet", index=False)
(RUTA_PROC / "resumen_preprocesamiento.json").write_text(
    json.dumps(resumen, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Guardado en:", RUTA_PROC)
for p in sorted(RUTA_PROC.iterdir()):
    print(f"  - {p.name} ({p.stat().st_size / 1e6:.1f} MB)")

Guardado en: D:\DSRP-ML3-Proyecto-Final\Datos\procesado
  - prod.parquet (18.3 MB)
  - resumen_preprocesamiento.json (0.0 MB)
  - train.parquet (169.1 MB)


## Análisis

- El dataset tiene 6.362.620 transacciones y solo el 0.13% son fraude. En el train (step <= 400) el fraude es del 0.08% y en producción (step >= 401) sube al 0.65%, o sea que el fraude se concentra más al final de la simulación.
- El fraude solo ocurre en TRANSFER (0.77% de esas transacciones) y CASH_OUT (0.18%); en PAYMENT, CASH_IN y DEBIT no hay ni uno. Por ello isTransfer e isCashOut le ayudan bastante al modelo.
- Se esperaba que los fraudes dejaran el saldo descuadrado, pero pasa lo contrario: en el 99.45% de los fraudes el saldo cuadra exacto porque el estafador se lleva el saldo completo de la cuenta. Las diferencias grandes de errorBalanceOrig salen en las transacciones normales (los CASH_IN suman saldo y los pagos cobran comisión). Lo que sí delata al fraude es que el monto casi iguala todo el saldo de la cuenta (ratio_amount cerca de 1).
- isFlaggedFraud atrapa solo 16 de 8.213 fraudes (0.19%), así que la saqué del modelo: la regla del negocio no sirve para esto.
- Las cuentas merchant (nameDest con M) siempre tienen saldo en 0; dejé isMerchant para que el modelo lo tenga en cuenta.
- Quedaron 12 features y los 5 folds estratificados guardados para el notebook de entrenamiento.